# Phase 2: Exploratory Data Analysis (EDA)

In this notebook, we'll analyze the PaySim dataset to understand its characteristics, uncover fraud patterns, and identify predictive features before moving on to modeling.

### Key Questions to Answer:
1. Which columns have nulls or anomalies?
2. What is the class imbalance ratio?
3. What is the fraud rate by transaction type?
4. What does the amount distribution look like for fraud vs legitimate?
5. Are there temporal patterns (fraud concentrated at certain hours)?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#1e1e1e", "figure.facecolor": "#1e1e1e", "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white"})

# Load the dataset
df = pd.read_csv('../data/raw/paysim_dataset.csv')
df.head()

## 1. Basic Dataset Info (Nulls & Types)
Let's check the shape of the data, the types of features, and whether any data is missing.

In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nMissing Values:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

> **Observation:** There are absolutely no missing values in this dataset. We have roughly 6.3 million rows and 11 columns.

## 2. Class Imbalance Ratio
Fraud detection datasets are famously imbalanced. Let's see the ratio of legitimate to fraudulent transactions.

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_ratio = fraud_counts[1] / len(df) * 100

print(f"Legitimate (0): {fraud_counts[0]}")
print(f"Fraudulent (1): {fraud_counts[1]}")
print(f"Fraud Ratio: {fraud_ratio:.3f}%")

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='isFraud', palette=['#3498db', '#e74c3c'])
plt.title('Distribution of Fraudulent vs Legitimate Transactions')
plt.yscale('log') # Log scale because of massive imbalance
plt.ylabel('Count (Log Scale)')
plt.show()

> **Observation:** The dataset is highly imbalanced! Only ~0.13% of transactions are fraud. This means we will need techniques like SMOTE or class-weighting (like `scale_pos_weight` in XGBoost) during training.

## 3. Fraud by Transaction Type
PaySim includes 5 transaction types. Let's see which types are vulnerable to fraud.

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.countplot(data=df, x='type', hue='isFraud', palette=['#3498db', '#e74c3c'])
plt.title('Transaction Counts by Type and Fraud Status')
plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.show()

# Calculate exact numbers
fraud_by_type = df.groupby('type')['isFraud'].agg(['count', 'sum'])
fraud_by_type.rename(columns={'count': 'Total', 'sum': 'Fraud_Count'}, inplace=True)
fraud_by_type['Fraud_Rate (%)'] = (fraud_by_type['Fraud_Count'] / fraud_by_type['Total']) * 100
display(fraud_by_type)

> **Crucial Finding:** Fraud *only* occurs in two transaction types: `TRANSFER` and `CASH_OUT`. This is a massive insight for feature engineering!

## 4. Amount Distribution (Fraud vs Legitimate)
Do fraudulent transactions tend to be larger or smaller than legitimate ones? Let's isolate `TRANSFER` and `CASH_OUT` since they contain all the fraud.

In [ ]:
vuln_df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=vuln_df, x='isFraud', y='amount', palette=['#3498db', '#e74c3c'], ax=axes[0])
axes[0].set_title('Transaction Amount by Fraud Status')
axes[0].set_yscale('log')

sns.kdeplot(data=vuln_df[vuln_df['isFraud'] == 0], x='amount', fill=True, color='#3498db', label='Legitimate', log_scale=True, ax=axes[1])
sns.kdeplot(data=vuln_df[vuln_df['isFraud'] == 1], x='amount', fill=True, color='#e74c3c', label='Fraud', log_scale=True, ax=axes[1])
axes[1].set_title('Amount Distribution (Log Scale)')
axes[1].legend()

plt.tight_layout()
plt.show()

> **Observation:** Fraudulent transactions often involve large amounts. There is also a notable density of fraud at exactly 0 amount, and also around the 1-10 million mark. `amount` is highly predictive.

## 5. Account Balances and "Emptying" Behavior
A common fraud pattern is transferring an exact amount to completely empty an account. Let's see if this happens here.

In [ ]:
vuln_df['balance_drop_pct'] = (vuln_df['oldbalanceOrg'] - vuln_df['newbalanceOrig']) / (vuln_df['oldbalanceOrg'] + 1)

plt.figure(figsize=(10, 6))
sns.histplot(data=vuln_df, x='balance_drop_pct', hue='isFraud', bins=50, palette=['#3498db', '#e74c3c'], stat='density', common_norm=False)
plt.title('Percentage of Balance Dropped during Transaction')
plt.xlabel('Balance Drop Ratio')
plt.show()

> **Observation:** For fraud cases, the balance drop ratio is almost exclusively at `1.0` (meaning the account was completely emptied). This is another extremely powerful feature!

## 6. Temporal Patterns (Hours & Days)
The `step` column represents 1 hour of time. Are fraudsters operating at specific times?

In [ ]:
df['hour_of_day'] = df['step'] % 24

plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='hour_of_day', hue='isFraud', palette=['#3498db', '#e74c3c'])
plt.title('Transactions by Hour of Day')
plt.yscale('log')
plt.show()

plt.figure(figsize=(12, 6))
fraud_only = df[df['isFraud'] == 1]
sns.histplot(data=fraud_only, x='step', bins=744, color='#e74c3c', kde=True) # 744 steps = 30 days
plt.title('Fraudulent Transactions over 30 Days (744 steps)')
plt.xlabel('Step (Hour)')
plt.show()

> **Observation:** While legitimate transactions follow a clear day/night cycle (dropping heavily during night hours), fraudulent transactions occur relatively consistently at *all* hours. Thus, a transaction occurring at 4 AM is highly suspicious.

## 7. Destination Balance Anomalies
Fraudsters often cash out into accounts with 0 initial balance, or immediately withdraw.

In [ ]:
vuln_df['dest_balance_increased'] = (vuln_df['newbalanceDest'] > vuln_df['oldbalanceDest']).astype(int)

plt.figure(figsize=(8, 5))
sns.barplot(data=vuln_df, x='isFraud', y='dest_balance_increased', palette=['#3498db', '#e74c3c'])
plt.title('Proportion of Transactions where Destination Balance Increased')
plt.ylabel('Proportion')
plt.show()

> **Observation:** In many fraudulent transactions, the destination balance doesn't seem to increase as expected (perhaps indicating complex multi-hop transfers or immediate withdrawals not captured in standard logging).

## 8. Flagged Fraud (`isFlaggedFraud`)
PaySim includes a baseline heuristic column `isFlaggedFraud` that flags transfers over 200,000. Let's see how effective it is.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print("Baseline Rule (Flag transfers > 200,000) Performance:\n")
print(classification_report(df['isFraud'], df['isFlaggedFraud']))

cm = confusion_matrix(df['isFraud'], df['isFlaggedFraud'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Legit', 'Predicted Fraud'], yticklabels=['Actual Legit', 'Actual Fraud'])
plt.title('Confusion Matrix: Baseline isFlaggedFraud Rule')
plt.show()

> **Observation:** The baseline `isFlaggedFraud` column is virtually useless. It catches almost nothing (recall is near 0%). This justifies the need for our Machine Learning models!

---

## Summary of EDA & Feature Engineering Plan

1. **Filter relevant data:** Fraud only happens in `TRANSFER` and `CASH_OUT`. We will focus our models strongly on these types (or one-hot encode the type).
2. **Handle Imbalance:** The 0.13% fraud rate requires `SMOTE` and algorithmic class weights.
3. **Key Engineered Features to build in Phase 4:**
   - `balance_drop_pct` (empty accounts)
   - `hour_of_day` (night vs day activity)
   - `amount_to_balance_ratio`
   - `dest_balance_increased`
4. **Windowed Features:** The steady stream of fraud means keeping track of velocity (e.g. `txn_count_1h`, `avg_amount_1h`) using our Faust streaming app will be extremely predictive.